# MahaTraffic AI — 02: Big Data Processing Pipeline
**Project:** MahaTraffic AI
**Focus:** Columnar Batch Analytics, Spark-Compatible Abstractions, and Parquet Performance
*Runs natively via Pandas/PyArrow with graceful PySpark integration.*


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

BASE_DIR = Path("..").resolve() if Path("..").resolve().name == "MahaTrafficAI" else Path(".").resolve()
print(f"Base Directory: {BASE_DIR}")


### 1. Parquet Metadata & Columnar File Stats
Inspect schema, row groups, and columnar metadata without loading the entire dataset into memory.


In [ ]:
parquet_path = BASE_DIR / "data" / "processed" / "parquet" / "accidents" / "maharashtra_accidents_clean.parquet"
meta = pq.read_metadata(parquet_path)
print(f"Number of rows: {meta.num_rows}")
print(f"Number of columns: {meta.num_columns}")
print(f"Number of row groups: {meta.num_row_groups}")
print(f"Serialized size: {meta.serialized_size} bytes")


### 2. Batch Aggregation & Analytical Risk Computation
Simulates distributed MapReduce / Spark batch aggregations using vectorised Pandas operations.


In [ ]:
df = pd.read_parquet(parquet_path)

# Group by district and compute analytical risk metric
agg_df = df.groupby("district").agg(
    total_accidents=("accident_count", "sum"),
    fatal_accidents=("fatal_accidents", "sum"),
    deaths=("deaths", "sum"),
    injuries=("injuries", "sum")
).reset_index()

# Multi-factor analytical risk index (0-100 normalized)
max_acc = agg_df["total_accidents"].max()
max_deaths = agg_df["deaths"].max()

agg_df["project_risk_score"] = (
    0.5 * (agg_df["total_accidents"] / max_acc * 100) +
    0.5 * (agg_df["deaths"] / max_deaths * 100)
).round(2)

agg_df["project_risk_level"] = agg_df["project_risk_score"].apply(
    lambda s: "HIGH" if s >= 60 else ("MEDIUM" if s >= 35 else "LOW")
)

agg_df.sort_values(by="project_risk_score", ascending=False).head(10)


### 3. Check for PySpark Environment
If PySpark is configured locally, we demonstrate PySpark DataFrame transformations; otherwise, explain fallback.


In [ ]:
try:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.appName("MahaTrafficAI-Local").master("local[2]").getOrCreate()
    spark_df = spark.read.parquet(str(parquet_path))
    print("PySpark initialized successfully.")
    spark_df.groupBy("year").sum("accident_count").show()
    spark.stop()
except Exception as e:
    print(f"Local PySpark not running or Java not installed (Normal for lightweight Windows setup): {e}")
    print("Production pipeline executes transparently using PyArrow / Pandas engine.")
